In [ ]:
!pip install pytorch-forecasting pytorch-lightning

import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

def prepare_tft_data(agg_sales):
    df = agg_sales.sort_values(['store', 'category_name', 'time_period']).copy()
    df['time_idx'] = df.groupby(['store', 'category_name']).cumcount()
    df['month'] = df['time_period'].dt.month
    df['year'] = df['time_period'].dt.year
    df['days_in_month'] = df['time_period'].dt.days_in_month
    return df

def train_tft_model(df):
    # Filter series with sufficient history
    min_time_steps = 6
    df = df.groupby(['store', 'category_name']).filter(lambda x: len(x) >= min_time_steps)
    
    # Create dataset
    max_encoder_length = 12
    max_prediction_length = 3
    training_cutoff = df['time_idx'].max() - max_prediction_length
    
    training = TimeSeriesDataSet(
        df[lambda x: x.time_idx <= training_cutoff],
        time_idx="time_idx",
        target="sale_dollars",
        group_ids=["store", "category_name"],
        static_categoricals=["store", "category_name"],
        time_varying_known_reals=["time_idx", "month", "year", "days_in_month"],
        time_varying_unknown_reals=["sale_dollars", "sale_bottles"],
        max_encoder_length=max_encoder_length,
        max_prediction_length=max_prediction_length,
        target_normalizer=GroupNormalizer(groups=["store", "category_name"], transformation="softplus"),
    )

    validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

    # Create dataloaders
    batch_size = 64
    train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
    val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

    # Configure TFT model
    pl.seed_everything(42)
    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=0.03,
        hidden_size=32,
        attention_head_size=2,
        dropout=0.1,
        hidden_continuous_size=16,
        output_size=7,
        loss=QuantileLoss(),
        log_interval=10,
        reduce_on_plateau_patience=4,
    )

    # Train model
    trainer = pl.Trainer(
        max_epochs=20,
        accelerator="auto",
        enable_model_summary=True,
        gradient_clip_val=0.1,
        callbacks=[
            EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min"),
            LearningRateMonitor()
        ],
    )

    trainer.fit(
        tft,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader,
    )
    
    return tft, training, validation

def predict_and_analyze(tft, training, df):
    # Generate predictions
    predictions = tft.predict(training.to_dataloader(train=False), return_y=True)
    
    # Process predictions
    results = []
    for idx, (prediction, _) in enumerate(zip(predictions, tft.predict(training.to_dataloader(train=False)))):
        raw_data = training.x_to_index(training.decode_batch(training.get(idx)))
        store = raw_data["store"][0]
        category = raw_data["category_name"][0]
        actuals = prediction.numpy()
        forecast = tft.predict(training.to_dataloader(train=False))[idx].numpy()
        
        # Calculate trend metrics
        pred_trend = np.diff(forecast)
        is_declining = np.all(pred_trend < 0)
        confidence = np.mean(np.abs(pred_trend)) / 100  # Normalized confidence
        
        results.append({
            "store": store,
            "category": category,
            "last_actual": actuals[-1],
            "forecast": forecast,
            "trend": pred_trend,
            "is_declining": is_declining,
            "confidence": min(1.0, confidence),
            "confidence_level": "High" if confidence > 0.7 else "Medium" if confidence > 0.4 else "Low"
        })
    
    return pd.DataFrame(results)

def plot_predictions(results_df, index=0):
    import matplotlib.pyplot as plt
    item = results_df.iloc[index]
    
    plt.figure(figsize=(12, 6))
    plt.plot(item['forecast'], label='Forecast')
    plt.title(f"Store: {item['store']} - Category: {item['category']}")
    plt.xlabel("Time Steps (Future)")
    plt.ylabel("Sales Dollars")
    plt.legend()
    plt.show()

if __name__ == "__main__":
    # Load and preprocess data
    file_path = 'sazerac_df.csv'
    df = pd.read_csv(file_path)
    df = preprocess_data(df)
    agg_sales = aggregate_store_product_sales(df)
    
    # Prepare TFT data
    tft_data = prepare_tft_data(agg_sales)
    
    # Train model
    tft, training, validation = train_tft_model(tft_data)
    
    # Generate predictions
    results_df = predict_and_analyze(tft, training, tft_data)
    
    # Show declining products
    declining_df = results_df[results_df['is_declining']].sort_values('confidence', ascending=False)
    print(f"Found {len(declining_df)} declining product-store combinations")
    print(declining_df[['store', 'category', 'confidence_level', 'last_actual']].head())
    
    # Plot sample prediction
    plot_predictions(results_df, index=0)